# Ollama na prática
## De uma mensagem de rádio a um dataset de pit wall

**DCA0305 · Machine Learning-Based Systems Design · UFRN**

> *"Rear left is gone. No grip in the last sector and the car behind is right on me. What are we doing?"*

Essa única mensagem de rádio é tudo de que este notebook trata. Na demonstração em sala ela entrou num `curl`. Aqui ela será resumida, transmitida token a token, lembrada, convertida em JSON, validada, garantida por um schema e, no fim, vira **uma linha de um DataFrame** ao lado de outras 29 mensagens. Se você conseguir seguir uma mensagem pelo pipeline inteiro, consegue seguir qualquer texto.

Um pouco de vocabulário para quem não acompanha Fórmula 1. O **pit wall** é o muro dos boxes, onde a equipe senta durante a corrida. O **race engineer** (engenheiro de pista) é quem fala com o piloto pelo rádio. Quando o piloto diz que o pneu **rear left** (traseiro esquerdo) "is gone", está dizendo que o pneu acabou, que não há mais aderência (*grip*), e a pergunta *"what are we doing?"* é um pedido de decisão, parar nos boxes ou aguentar. Ao longo da aula, um modelo de linguagem rodando no seu computador fará o papel desse engenheiro.


### 🎯 Objetivos de aprendizagem

Ao final desta aula, você será capaz de

1. **Explicar** o que o Ollama é (um servidor HTTP na mesma máquina que o notebook) e o que ele não é (uma biblioteca Python).
2. **Conversar** com um modelo local por duas portas, o HTTP cru com `requests` e o SDK da OpenAI com a `base_url` trocada, e dizer quando usar cada uma.
3. **Controlar** a aleatoriedade com `temperature` e `seed`, e apontar qual dos dois garante reprodutibilidade.
4. **Medir** tokens por segundo pela API nativa e usar esse número numa decisão de produto.
5. **Implementar** memória de conversa e explicar por que ela custa tokens a cada chamada.
6. **Forçar** saídas estruturadas em três níveis (modo JSON, contrato Pydantic, *structured outputs*) e dizer exatamente o que cada nível garante e o que não garante.
7. **Transformar** 30 mensagens de texto em um DataFrame e num gráfico que segue as regras da aula 04.

## 📐 Como estudar este notebook

Este material foi feito para ser estudado **sem professor por perto**. O notebook tem um ritmo fixo e um contrato explícito.

### O ritmo de cada bloco

Todo bloco tem as mesmas seis partes, na mesma ordem, com os mesmos ícones. Aprenda o ritmo uma vez e nunca mais precisará pensar no layout, só no conteúdo.

| Parte | O que é | O que você faz |
|---|---|---|
| 🎯 **Uma ideia** | Uma frase. A única coisa nova do bloco. | Leia. |
| 🔮 **Preveja** | Uma pergunta cuja resposta a próxima célula revela. | Escreva a sua aposta **antes** de rodar. No papel ou na própria célula. |
| ▶️ **Rode** | Uma célula completa, que funciona como está, seguida de 🔍 **O que você deve ver**. | Rode, leia a saída com calma, compare com a aposta. |
| 🧩 **Preencha a lacuna** | Uma célula com um buraco marcado `# ---- SEU CÓDIGO AQUI ----`, acompanhada de 🧪 **Como saber se deu certo** e de uma 🔑 **solução de referência** escondida. | Complete antes de seguir. Abra a solução só depois de tentar de verdade. |
| 🔒 **Fechamento** | Uma frase que você deve conseguir dizer em voz alta. | Diga. Em voz alta mesmo. |
| 🧭 **Por que o experimento é assim** e 🐇 **Toca do coelho** | Uma nota curta sobre o desenho do bloco, e uma trilha opcional para quem quer ir além. | Leia a nota. Entre na toca se tiver curiosidade e tempo. |

Fechando cada bloco há um 📓 **Diário de bordo**, uma célula sua com duas linhas para preencher.

### O contrato

1. **Aposte antes de rodar.** Cada 🔮 pede uma previsão. Errar a previsão e corrigir é o que grava o conteúdo. É o efeito de geração, o mesmo do 🔮 Predict da aula 04.
2. **A regra dos dez minutos.** Só abra uma 🔑 solução depois de dez minutos tentando. Se abrir antes, você troca aprendizagem por alívio.
3. **Quebre as coisas.** Toda célula ▶️ é um ponto de partida, não um ponto final. Mude um parâmetro, troque o modelo, apague uma linha e veja o que acontece. O modelo roda na máquina virtual do Colab, não há conta a pagar e nada que você faça aqui quebra algo de verdade.
4. **Duas sessões, não uma.** Faça os Blocos 0 a 3 num dia e os Blocos 4 a 7 em outro. O intervalo faz parte do método (prática espaçada), não é uma concessão. No Colab, cada sessão começa do zero, então as células de preparação (instalar o Ollama, subir o servidor, baixar os modelos) rodam de novo a cada vez. São uns três minutos.
5. **Escreva no diário.** Duas linhas por bloco. O que me surpreendeu. O que eu testaria a seguir. Escrever sobre o que você acabou de fazer é a forma mais barata de metacognição que existe.

Tempo estimado. Cerca de 90 minutos por sessão.

## 🧰 Preparação (no próprio Colab)

No Colab, o Ollama não vem instalado, e a máquina virtual é apagada quando a sessão termina. Por isso a preparação é feita por células, aqui mesmo, e precisa ser repetida a cada sessão. São três passos e uns três minutos.

Antes de rodar, uma escolha que muda tudo. Vá em **Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU**. Com a GPU, o modelo grande responde em segundos. Sem ela (CPU, dois núcleos), tudo funciona, só mais devagar, e o texto de cada bloco traz as duas faixas de tempo. Se a GPU não estiver disponível no momento, siga em CPU sem medo.

| Passo | O que faz | Quanto tempo |
|---|---|---|
| 1 · Instalar | Instala o descompactador `zstd` e baixa o programa `ollama` para a máquina virtual | cerca de 1 minuto |
| 2 · Subir o servidor | Inicia o `ollama serve` em segundo plano e espera ele responder | segundos |
| 3 · Baixar os modelos | `ollama pull` do pequeno (~400 MB), do grande (~2 GB) e do SmolLM2 (~270 MB) | 1 a 2 minutos |

Se a máquina virtual não aguentar o modelo grande (raro no Colab, que tem cerca de 12 GB de RAM), faça toda a aula com `BIG = SMALL` na célula de configuração. Nada quebra. Você só perde a comparação entre tamanhos.

In [ ]:
# Passo 1 · Instala o Ollama na máquina virtual do Colab. O script oficial detecta a GPU, se houver.
# O instalador entrega o Ollama num pacote .tar.zst e exige o descompactador zstd, que o Colab não traz por padrão.
!apt-get install -y -qq zstd > /dev/null 2>&1 || (apt-get update -qq > /dev/null 2>&1 && apt-get install -y -qq zstd > /dev/null 2>&1)
!zstd --version
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -n 5
!ollama --version
# O aviso "could not connect to a running Ollama instance" é esperado: o programa existe, o servidor ainda não. É o passo 2.
# Se em vez disso aparecer "ollama: command not found", o instalador não terminou. Rode a célula de novo sem o "| tail -n 5" para ver a mensagem inteira.

In [ ]:
# Passo 2 · Sobe o servidor em segundo plano e espera até ele responder.
import subprocess, time, requests

OLLAMA_URL = "http://localhost:11434"

def server_is_up() -> bool:
    try:
        return requests.get(f"{OLLAMA_URL}/api/version", timeout=2).ok
    except requests.RequestException:             # sem conexão ou sem resposta a tempo
        return False

if not server_is_up():
    subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT,
                     start_new_session=True)          # continua vivo depois que a célula termina
    for _ in range(30):                               # espera até 30 s pelo servidor
        if server_is_up():
            break
        time.sleep(1)

if server_is_up():
    print("servidor de pé em", OLLAMA_URL, "·", requests.get(f"{OLLAMA_URL}/api/version").json())
else:
    print("o servidor não respondeu. Veja o arquivo ollama.log (aba de arquivos, à esquerda).")

In [ ]:
# Passo 3 · Baixa os modelos. O progresso aparece abaixo. Se a conexão cair, rode a célula de novo (o download continua de onde parou).
!ollama pull qwen2.5:0.5b
!ollama pull qwen2.5:3b
!ollama pull smollm2:360m
!ollama list

In [ ]:
# Instala o que falta. Rode uma vez por ambiente. O -q deixa a saída silenciosa.
!pip -q install openai pydantic requests pandas matplotlib

In [ ]:
# Configuração compartilhada por todo o notebook. Rode uma vez.
import json, time, os, requests
from typing import Literal, Optional
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError
import pandas as pd
import matplotlib.pyplot as plt

OLLAMA_URL = "http://localhost:11434"      # o servidor, na máquina virtual do Colab. O mesmo endereço do curl da demo.
SMALL = "qwen2.5:0.5b"                     # o modelo pequeno (~400 MB em disco)
BIG   = "qwen2.5:3b"                       # o modelo grande (~2 GB). Troque por SMALL se a máquina sofrer.

# A mensagem que atravessa o notebook inteiro (em inglês, porque rádio de F1 é em inglês
# e porque modelos pequenos rendem mais nele; o Bloco 5 testa a versão em português).
RADIO = ("Rear left is gone. No grip in the last sector and the car behind "
         "is right on me. What are we doing?")
RADIO_PT = ("O traseiro esquerdo já era. Sem aderência no último setor e o carro de trás "
            "está colado em mim. O que a gente vai fazer?")

# Aperto de mão. Se esta linha falhar, o servidor não está de pé (volte ao passo 2 da preparação).
print("Ollama respondeu:", requests.get(f"{OLLAMA_URL}/api/version", timeout=5).json())

---
# Bloco 0 · Aperto de mão

### 🎯 Uma ideia
**O Ollama não é uma biblioteca. É um servidor HTTP rodando na mesma máquina que este notebook, e tudo nesta aula é uma requisição para ele.**

Vale a pena parar nessa frase. Quando o passo 1 da preparação instalou o Ollama, ele não instalou um pacote Python. Instalou um programa que, desde o passo 2, fica escutando na porta 11434 da máquina virtual, carrega modelos na memória quando alguém pede e responde em JSON. O aplicativo de desktop que você viu na demo, a linha de comando `ollama run`, o `curl` e este notebook são apenas quatro clientes diferentes falando com o mesmo tipo de servidor. Aqui a máquina é do Google e não a sua, mas para o notebook isso não muda nada, o endereço continua sendo `localhost`. Se você entender isso, o resto da aula é detalhe de sintaxe.

Há duas portas de entrada para esse servidor, e você vai usar as duas.

| Porta | Como | Para quê |
|---|---|---|
| **Porta 1 · HTTP cru** | `requests.get(...)` e `requests.post(...)` nos endpoints `/api/...` | Ver tudo o que o Ollama sabe (tamanhos, quantização, velocidade em nanossegundos) |
| **Porta 2 · SDK da OpenAI** | `OpenAI(base_url="http://localhost:11434/v1")` | Escrever código que funciona igual com um modelo local ou com um modelo na nuvem |

Neste bloco, só a Porta 1.

### 🔮 Preveja
Antes de rodar a célula, escreva as duas apostas.

1. Quantos modelos a lista vai mostrar, e qual deles ocupa mais espaço em disco?
2. O `qwen2.5:0.5b` tem cerca de 500 milhões de parâmetros. Se cada parâmetro fosse um `float32` (4 bytes), quanto ele ocuparia? E quanto você aposta que ele ocupa de verdade no seu disco?

### ▶️ Rode

In [ ]:
# Porta 1: HTTP cru. É o curl da demo traduzido para Python, linha por linha.
r = requests.get(f"{OLLAMA_URL}/api/tags")
r.raise_for_status()                         # levanta um erro se a resposta não for 2xx

models = r.json()["models"]                  # uma lista de dicionários, um por modelo instalado
print(f"{'modelo':<16} {'disco (GB)':>10} {'quantização':>12} {'família':>8} {'parâmetros':>11}")
for m in models:
    d = m["details"]
    print(f"{m['name']:<16} {m['size']/1e9:>10.2f} {d['quantization_level']:>12} "
          f"{d['family']:>8} {d['parameter_size']:>11}")

In [ ]:
# A ficha técnica do modelo pequeno. Compare cada linha com as sete paradas da aula 06.
info = requests.post(f"{OLLAMA_URL}/api/show", json={"model": SMALL}).json()
mi = info["model_info"]

def find(suffix):
    # As chaves vêm prefixadas pela arquitetura (ex.: "qwen2.context_length"),
    # então procuramos pelo sufixo para o código funcionar com qualquer modelo.
    return next((v for k, v in mi.items() if k.endswith(suffix)), None)

print("arquitetura        ", mi.get("general.architecture"))
print("parâmetros         ", f"{mi.get('general.parameter_count', 0):,}")
print("blocos (camadas)   ", find("block_count"))
print("largura d          ", find("embedding_length"))
print("cabeças de atenção ", find("attention.head_count"), "| cabeças K/V", find("attention.head_count_kv"))
print("janela de contexto ", find("context_length"), "tokens")
print("capacidades        ", info.get("capabilities"))

### 🔍 O que você deve ver

Na primeira célula, uma tabela com os modelos que você baixou. Repare em três colunas.

- **disco (GB).** O `qwen2.5:0.5b` ocupa cerca de 0,40 GB. Se você apostou em 4 bytes por parâmetro, esperava ~2 GB. A diferença é a próxima coluna.
- **quantização.** `Q4_K_M` é o formato padrão da biblioteca do Ollama. O "4" diz que a maior parte dos pesos foi comprimida para cerca de 4 bits. É a mesma matriz de pesos da aula 06, só que cada número ocupa um oitavo do espaço de um `float32`. A perda de qualidade é pequena e o ganho de memória e velocidade é enorme. É por isso que um modelo de 3 bilhões de parâmetros cabe num notebook comum.
- **parâmetros.** `494.03M` para o pequeno e `3.09B` para o grande. Guarde esses dois números, a lacuna abaixo usa os dois.

Na segunda célula, a ficha técnica que o `/api/show` devolve. Cada linha é uma parada da aula 06. `block_count` é o número de blocos Transformer empilhados. `embedding_length` é a largura `d` de cada vetor. `context_length` é o tamanho da tabela de posições, o limite físico de tokens que o modelo consegue olhar de uma vez. E `head_count_kv` menor que `head_count` é a *grouped-query attention*, uma das mudanças de 2019 para 2024 que você anotou no relatório de seleção de modelos.

Se a família aparecer como `qwen2` e não `qwen2.5`, está certo. A arquitetura é a mesma da geração anterior, o que mudou foi o treinamento.

### 🧩 Preencha a lacuna

Toda tag de modelo esconde um nível de quantização. Calcule, para `SMALL` e para `BIG`, **quantos bytes o arquivo gasta por parâmetro**. Você tem tudo o que precisa em `models`, o tamanho em bytes em `size` e a contagem de parâmetros em `details["parameter_size"]`, que vem como texto (`"494.03M"`, `"3.09B"`).

✋ **Sua escolha.** Faça o cálculo só para os dois modelos da aula, ou escreva a função de forma que ela funcione para *qualquer* modelo instalado (inclusive um que use sufixo `K`).

🧪 **Como saber se deu certo.** Um modelo `Q4_K_M` deveria dar algo perto de 0,5 byte por parâmetro (4 bits). O pequeno vai dar **mais** que isso, na casa de 0,8, e o grande vai dar menos, na casa de 0,6. Se os seus números estiverem nessa faixa, o cálculo está certo. Se der 4, você esqueceu a quantização. Se der 0,0008, esqueceu o sufixo `M`.

In [ ]:
def parse_params(s: str) -> float:
    # "494.03M" -> 494_030_000 ; "3.09B" -> 3_090_000_000
    # ---- SEU CÓDIGO AQUI ----
    ...

def bytes_per_parameter(name: str) -> float:
    # 1. localize o modelo em `models` pelo nome
    # 2. divida o tamanho em bytes pela contagem de parâmetros
    # ---- SEU CÓDIGO AQUI ----
    ...

for name in (SMALL, BIG):
    bpp = bytes_per_parameter(name)
    print(f"{name:<14} {bpp:.2f} bytes por parâmetro  (~{bpp*8:.1f} bits)")

<details>
<summary><b>🔑 Solução de referência</b> (abra só depois dos dez minutos)</summary>

```python
def parse_params(s: str) -> float:
    mult = {"K": 1e3, "M": 1e6, "B": 1e9}
    return float(s[:-1]) * mult[s[-1].upper()]

def bytes_per_parameter(name: str) -> float:
    m = next(m for m in models if m["name"] == name)
    return m["size"] / parse_params(m["details"]["parameter_size"])

for name in (SMALL, BIG):
    bpp = bytes_per_parameter(name)
    print(f"{name:<14} {bpp:.2f} bytes por parâmetro  (~{bpp*8:.1f} bits)")
```

**Por que o pequeno gasta mais bytes por parâmetro?** Duas razões, e as duas vêm da aula 06. Primeiro, `Q4_K_M` não é "tudo em 4 bits". É uma mistura. Alguns tensores (a saída e partes da atenção) ficam em 6 bits para proteger a qualidade. Segundo, a tabela de embeddings é guardada com mais precisão e, num modelo de 0,5B com vocabulário de ~150 mil tokens, essa tabela é mais de um quarto de todos os parâmetros (parada 7, contados à mão). Num modelo de 3B, a mesma tabela é uma fração bem menor do total. Quantização é uma média ponderada, e o peso de cada tensor muda com o tamanho do modelo.

</details>

### 🔒 Fechamento
**Se dá para fazer `curl`, dá para programar. O notebook só troca o `curl` por Python.**

### 🧭 Por que o experimento é assim
Este bloco é o **figura e fundo** da aula inteira. Antes de qualquer modelo responder qualquer coisa, você precisava ver o servidor como servidor, uma coisa que existe independentemente do notebook. Por isso a primeira chamada não gera texto nenhum, ela só lista arquivos. Também é o bloco mais ancorado no que você já viu (o `curl` da demo, a contagem de parâmetros da aula 06), o que Ausubel chamaria de *organizador prévio*. A lacuna parece uma conta boba, mas ela obriga você a abrir o JSON com as próprias mãos e a converter um texto como `"494.03M"` num número, exatamente o tipo de detalhe que ninguém lembra de ter lido e todo mundo lembra de ter feito.

### 🐇 Toca do coelho
Duas trilhas.

1. Rode `!ollama ps` numa célula (ou `requests.get(f"{OLLAMA_URL}/api/ps").json()`) agora, antes de qualquer geração. Depois de rodar a primeira célula do Bloco 1, rode de novo. O que apareceu? Repare no campo `expires_at`. O Ollama mantém o modelo carregado na memória por cinco minutos depois da última chamada (`keep_alive`) e descarrega em seguida. Isso explica por que a primeira chamada de cada bloco é mais lenta que a segunda, e vai reaparecer como `load_duration` no Bloco 2.
2. Rode `!ollama show --modelfile qwen2.5:0.5b` numa célula (a mesma informação está em `info["modelfile"]` e `info["template"]`). Todo modelo do Ollama nasce de um **Modelfile**, uma receita com o arquivo de pesos (`FROM`), o molde que transforma a lista de mensagens em texto (`TEMPLATE`), um `SYSTEM` opcional e `PARAMETER`s como a temperatura. Leia o do modelo pequeno com calma. No exercício da semana você vai escrever o seu.

In [ ]:
# 🐇 Espaço livre para a toca do coelho. Nada aqui é obrigatório.

### 📓 Diário de bordo · Bloco 0

*(clique duas vezes nesta célula e escreva duas linhas; ninguém vai avaliar a gramática)*

- **O que me surpreendeu.** ...
- **O que eu testaria a seguir.** ...

---
# Bloco 1 · Texto cru

### 🎯 Uma ideia
**O mesmo prompt não dá a mesma resposta duas vezes, e `temperature` é o botão que você já conheceu na parada 6 da aula 06.**

Lembre do desenho. Depois do último bloco, o modelo produz 150 mil pontuações, uma por token do vocabulário. A softmax transforma as pontuações numa distribuição e um dado é rolado. A temperatura divide as pontuações antes da softmax. Temperatura alta achata a distribuição e o dado fica mais "justo". Temperatura baixa afia a distribuição e o dado quase sempre cai no token mais provável. Temperatura zero é o caso limite, sempre o token mais provável, sem dado nenhum.

Neste bloco entra a **Porta 2**, o SDK da OpenAI apontado para o servidor local. Repare que o código abaixo é *idêntico* ao que você escreveria para falar com um modelo na nuvem. Muda uma linha, a `base_url`. Essa é a razão de a indústria ter convergido para esse formato de API, e é a razão de a aula usar o SDK em vez de um pacote específico do Ollama.

### 🔮 Preveja
1. Com `temperature=1.0`, as três respostas serão idênticas, parecidas ou diferentes?
2. Em que valor de temperatura você aposta que as três respostas ficam **exatamente** iguais?
3. Existe outro parâmetro, além da temperatura, que torna a resposta reproduzível? Qual?

### ▶️ Rode

In [ ]:
# Porta 2: o SDK da OpenAI apontado para o servidor local. É a porta do resto da aula.
client = OpenAI(base_url=f"{OLLAMA_URL}/v1", api_key="ollama")   # a chave é exigida pelo SDK e ignorada pelo Ollama

PROMPT = f"Summarize what the driver is asking for in one sentence.\n\nRadio: {RADIO}"

for i in range(3):
    resp = client.chat.completions.create(
        model=SMALL,
        messages=[{"role": "user", "content": PROMPT}],
        temperature=1.0,
    )
    print(f"[{i+1}] {resp.choices[0].message.content.strip()}\n")

### 🔍 O que você deve ver

Três frases diferentes sobre o mesmo pedido. Em geral o modelo acerta o essencial (o piloto quer saber o que a equipe vai fazer sobre o pneu) e varia na forma. Às vezes ele inventa um detalhe que não está na mensagem, um número de voltas, um nome de pneu. Esse é o comportamento normal de um modelo de 0,5B com temperatura 1. Não é um defeito da instalação.

Repare também no objeto que voltou. `resp.choices[0].message.content` é o texto. `resp.usage` diz quantos tokens entraram e saíram. Esse formato de resposta é o mesmo de qualquer provedor compatível com a API da OpenAI, e é por isso que ele vale a pena aprender uma vez só.

### 🧩 Preencha a lacuna

Complete a função `same_answer`, que faz `n` chamadas iguais e devolve se todas as respostas foram idênticas. Depois rode o experimento em duas partes.

- **Parte A, sem `seed`.** Temperaturas `1.0, 0.7, 0.3, 0.0`, três chamadas cada. Em qual delas as respostas ficam idênticas?
- **Parte B, com `seed=42`.** As mesmas temperaturas. O que muda?

✋ **Sua escolha.** Use o `PROMPT` acima ou escreva o seu próprio, desde que peça algo com margem para variação (um resumo, uma resposta ao piloto). Um prompt que só admite uma resposta certa ("what is 2+2") esconde o fenômeno.

🧪 **Como saber se deu certo.** Na Parte A, só a temperatura `0.0` deve dar `idênticas`. Na Parte B, **todas** devem dar `idênticas`, inclusive `1.0`. Se a Parte B não ficou toda idêntica, confira se o `seed` está sendo passado em todas as chamadas. Se a Parte A já deu idênticas em `0.3`, não é erro, é a distribuição afiada demais para o dado cair em outro lugar em três tentativas. Rode com `n=6`.

In [ ]:
def same_answer(model: str, prompt: str, temperature: float, seed: Optional[int] = None, n: int = 3):
    """Faz n chamadas iguais e devolve (todas_iguais: bool, respostas: list[str])."""
    answers = []
    for _ in range(n):
        # ---- SEU CÓDIGO AQUI ----
        # monte a chamada; passe seed=seed apenas quando seed não for None
        ...
    return len(set(answers)) == 1, answers

for label, seed in (("Parte A · sem seed", None), ("Parte B · seed=42", 42)):
    print(label)
    for T in (1.0, 0.7, 0.3, 0.0):
        identical, answers = same_answer(SMALL, PROMPT, T, seed)
        print(f"  T={T:<4} {'idênticas' if identical else 'diferentes'}")
    print()

<details>
<summary><b>🔑 Solução de referência</b></summary>

```python
def same_answer(model: str, prompt: str, temperature: float, seed: Optional[int] = None, n: int = 3):
    answers = []
    for _ in range(n):
        kwargs = dict(model=model,
                      messages=[{"role": "user", "content": prompt}],
                      temperature=temperature)
        if seed is not None:
            kwargs["seed"] = seed
        resp = client.chat.completions.create(**kwargs)
        answers.append(resp.choices[0].message.content.strip())
    return len(set(answers)) == 1, answers

for label, seed in (("Parte A · sem seed", None), ("Parte B · seed=42", 42)):
    print(label)
    for T in (1.0, 0.7, 0.3, 0.0):
        identical, answers = same_answer(SMALL, PROMPT, T, seed)
        print(f"  T={T:<4} {'idênticas' if identical else 'diferentes'}")
    print()
```

**O que a Parte B ensina.** O dado da parada 6 é um gerador de números pseudoaleatórios. Com o mesmo `seed`, o mesmo prompt e os mesmos parâmetros, o gerador produz a mesma sequência de "sorteios" e o texto sai idêntico mesmo com temperatura 1. Ou seja, **temperatura controla diversidade, `seed` controla reprodutibilidade**, e são coisas diferentes. Uma demo reproduzível usa os dois, temperatura baixa para reduzir invenções e `seed` fixo para que o resultado não mude entre uma execução e outra.

</details>

### 🔒 Fechamento
**Amostragem é uma escolha, não um acidente. Temperatura zero mais `seed` fixo é como se faz uma demo reproduzível.**

### 🧭 Por que o experimento é assim
O bloco é um ciclo **prever, observar, explicar** puro. A previsão da pergunta 2 costuma ser "temperatura zero", e está certa, mas incompleta, e a Parte B existe para mostrar a segunda metade da resposta com um resultado que contradiz a intuição (respostas idênticas com temperatura 1). Um resultado que surpreende gera mais retenção do que um que confirma, e é isso que o Loewenstein descreve como fechar uma lacuna de informação. Na Gestalt, é **Prägnanz**. A conclusão mais simples que explica as duas partes ("`seed` reproduz, temperatura diversifica") é a que fica.

### 🐇 Toca do coelho
Dois caminhos.

1. `temperature=0` nem sempre é determinística entre máquinas diferentes, ou entre uma chamada isolada e uma chamada que dividiu o servidor com outras. Aritmética de ponto flutuante em paralelo não é associativa, e a ordem das somas muda o último bit de um logit, que às vezes muda o token escolhido. Procure por "batch invariance" para ver como esse problema vira um tema de engenharia em produção.
2. Além da temperatura há `top_p` e `top_k`, que cortam a cauda da distribuição antes do sorteio. O SDK aceita `top_p` direto. Já `top_k` é um parâmetro nativo do Ollama e só entra pela Porta 1, no campo `"options"` do `POST /api/chat` (o endpoint compatível com a OpenAI ignora silenciosamente o que não conhece, teste e veja). Compare `top_k=1` com `temperature=0`. Deveriam ser equivalentes. São?

In [ ]:
# 🐇 Espaço livre para a toca do coelho. Nada aqui é obrigatório.

### 📓 Diário de bordo · Bloco 1

*(clique duas vezes nesta célula e escreva duas linhas; ninguém vai avaliar a gramática)*

- **O que me surpreendeu.** ...
- **O que eu testaria a seguir.** ...

---
# Bloco 2 · Fluxo

### 🎯 Uma ideia
**O modelo produz um token de cada vez (parada 6 da aula 06). O *streaming* deixa você assistir, e a API nativa diz a que velocidade.**

Você já escreveu esse laço à mão, `generate_by_hand`, na aula 06. Rode o modelo, sorteie um token, acrescente ao texto, rode de novo. O que o servidor faz com `stream=True` é simplesmente enviar cada token pela rede assim que ele é sorteado, em vez de esperar o laço terminar. Nada muda no modelo. O que muda é a experiência de quem espera. Uma resposta de 80 tokens a 20 tokens por segundo leva 4 segundos. Sem streaming, o usuário olha 4 segundos para uma tela vazia. Com streaming, começa a ler no primeiro décimo de segundo.

Isso introduz as duas métricas que decidem se um modelo pode viver dentro de um produto. O **tempo até o primeiro token** (*time to first token*, TTFT), que é o quanto o usuário espera para ver alguma coisa, e os **tokens por segundo** de geração, que é o quanto ele espera pelo resto. A porta 2 mostra o texto chegando. A porta 1 entrega os números.

### 🔮 Preveja
1. Quantos tokens por segundo o `qwen2.5:0.5b` vai gerar no Colab? E o `qwen2.5:3b`? Escreva os dois números (e anote se está em CPU ou em GPU).
2. A razão entre os dois será próxima da razão entre os tamanhos (3B / 0,5B ≈ 6), maior ou menor?
3. Uma pessoa lê em voz alta cerca de 2 a 3 palavras por segundo. Qual dos dois modelos é mais rápido do que uma pessoa lendo?

### ▶️ Rode

In [ ]:
# stream=True: os pedaços chegam à medida que o modelo os produz. Repare no ritmo da impressão.
t0 = time.time()
first_token_at = None
n_chunks = 0

stream = client.chat.completions.create(
    model=SMALL,
    messages=[{"role": "user",
               "content": f"Reply to this driver as a calm race engineer, in two sentences: {RADIO}"}],
    stream=True,
)
for chunk in stream:
    if not chunk.choices:                        # alguns pedaços podem vir sem conteúdo
        continue
    piece = chunk.choices[0].delta.content or ""  # o último pedaço costuma vir vazio
    if first_token_at is None and piece:
        first_token_at = time.time() - t0         # tempo até o primeiro token (TTFT)
    print(piece, end="", flush=True)              # flush=True força a impressão imediata
    n_chunks += 1

total = time.time() - t0
print(f"\n\n{n_chunks} pedaços · primeiro token em {first_token_at:.2f} s · total {total:.2f} s")

### 🔍 O que você deve ver

O texto aparecendo aos pedaços, e não de uma vez. Cada pedaço é um token, e um token nem sempre é uma palavra. Preste atenção em palavras longas ou raras chegando em partes (`ty` + `res`, `under` + `cut`). É a parada 1 da aula 06 acontecendo ao vivo.

A linha final resume a experiência do usuário. O tempo até o primeiro token inclui o custo de **ler o prompt** inteiro (o *prefill*) e, se o modelo ainda não estava na memória, o custo de **carregá-lo** do disco. Rode a célula uma segunda vez e veja o primeiro token chegar bem mais cedo. O modelo já estava carregado (lembra do `ollama ps` da toca do Bloco 0?).

### 🧩 Preencha a lacuna

O SDK esconde a velocidade. A Porta 1 não. Uma chamada a `POST /api/chat` com `"stream": false` devolve, junto com a resposta, um conjunto de contadores em **nanossegundos**.

| Campo | O que mede |
|---|---|
| `prompt_eval_count`, `prompt_eval_duration` | Tokens do prompt e o tempo para lê-los (*prefill*) |
| `eval_count`, `eval_duration` | Tokens gerados e o tempo para gerá-los (*decode*) |
| `load_duration` | Tempo para carregar o modelo, se ele não estava na memória |
| `total_duration` | Tudo somado |

Complete `speed()` para devolver tokens por segundo de geração e de leitura do prompt, mais o tempo de carga em segundos. Rode para `SMALL` e para `BIG` e **anote os dois números de geração**. Você vai precisar deles no exercício da semana.

✋ **Sua escolha.** Medir uma vez é medir ruído. Rode cada modelo duas vezes e guarde a segunda (com o modelo já carregado), ou rode três vezes e tire a mediana. Decida e justifique numa linha de comentário.

🧪 **Como saber se deu certo.** No Colab em CPU (dois núcleos), o pequeno fica na casa de 8 a 25 tokens por segundo e o grande na de 2 a 8. Com a GPU T4, o pequeno passa de 100 e o grande fica entre 50 e 100. O `prompt_tok_s` deve ser bem maior que o `gen_tok_s` (ler é paralelo, gerar é sequencial). Se `gen_tok_s` der na casa dos milhões, você esqueceu de converter nanossegundos em segundos. Se der zero ou o código levantar `KeyError`, o modelo ainda estava carregando e a resposta veio incompleta; rode de novo.

In [ ]:
def speed(model: str, prompt: str) -> dict:
    """Mede a velocidade pela API nativa (Porta 1). Tempos chegam em nanossegundos."""
    payload = {"model": model, "stream": False,
               "messages": [{"role": "user", "content": prompt}]}
    r = requests.post(f"{OLLAMA_URL}/api/chat", json=payload).json()
    # ---- SEU CÓDIGO AQUI ----
    # gen_tok_s    = eval_count / eval_duration (em segundos)
    # prompt_tok_s = prompt_eval_count / prompt_eval_duration (em segundos)
    # load_s       = load_duration em segundos
    return {"model": model, "gen_tok_s": ..., "prompt_tok_s": ..., "load_s": ...}

rows = []
for m in (SMALL, BIG):
    ...   # sua estratégia de medição (uma vez? duas, guardando a segunda? mediana de três?)
pd.DataFrame(rows).round(1)

<details>
<summary><b>🔑 Solução de referência</b></summary>

```python
def speed(model: str, prompt: str) -> dict:
    payload = {"model": model, "stream": False,
               "messages": [{"role": "user", "content": prompt}]}
    r = requests.post(f"{OLLAMA_URL}/api/chat", json=payload).json()
    ns = 1e9
    return {"model": model,
            "gen_tok_s": r["eval_count"] / (r["eval_duration"] / ns),
            "prompt_tok_s": r["prompt_eval_count"] / (r["prompt_eval_duration"] / ns),
            "load_s": r["load_duration"] / ns}

rows = []
for m in (SMALL, BIG):
    speed(m, RADIO)                 # aquecimento: carrega o modelo e não conta
    rows.append(speed(m, RADIO))    # a medição que vale, com o modelo já na memória
pd.DataFrame(rows).round(1)
```

Sobre a pergunta 2 do 🔮. A razão de velocidade costuma ser **menor** que a razão de tamanhos. Numa CPU, gerar um token é limitado pela largura de banda de memória (o modelo inteiro passa pela memória a cada token), e um modelo 6 vezes maior não é 6 vezes mais lento porque parte do custo (carregar o prompt, a amostragem, o servidor) não cresce com o tamanho. Meça e confira.

</details>

### 🔒 Fechamento
**Tokens por segundo é o número que decide se um modelo cabe dentro de um produto. Um modelo de 0,5B e um de 3B diferem em muito mais do que qualidade.**

### 🧭 Por que o experimento é assim
A lacuna é um **experimento controlado**. A mesma mensagem, a mesma função, a mesma máquina virtual, e só uma variável muda, o tamanho do modelo. É o desenho mais simples que produz uma conclusão defensável, e é o mesmo desenho que o seu relatório da semana precisa ter. Na Gestalt, é **similaridade** posta a serviço da comparação. Duas linhas de uma tabela com as mesmas colunas são lidas como um par, e a diferença entre elas salta aos olhos sem esforço. O ✋ existe porque decidir *como* medir (uma vez, duas, mediana) é uma decisão de engenharia de verdade, e não há resposta certa sem saber para que a medida vai servir.

### 🐇 Toca do coelho
Duas trilhas.

1. **Prefill contra decode.** Aumente o prompt (cole a mensagem dez vezes) e meça de novo. `prompt_tok_s` deve continuar alto e o TTFT deve subir. Agora você sabe por que APIs comerciais cobram tokens de entrada mais barato do que tokens de saída, e por que "modo raciocínio" (que gera milhares de tokens antes de responder) custa tanto.
2. **O engenheiro dentro do carro.** Se um dia esse engenheiro virar uma voz dentro de um carro de verdade, a resposta precisa vir antes de a pergunta perder o sentido. Um pneu que "is gone" não espera 10 segundos. Com os seus números, qual dos dois modelos aguenta esse cenário? E o que muda se a resposta for lida em voz alta por um sintetizador (2 a 3 palavras por segundo)?

In [ ]:
# 🐇 Espaço livre para a toca do coelho. Nada aqui é obrigatório.

### 📓 Diário de bordo · Bloco 2

*(clique duas vezes nesta célula e escreva duas linhas; ninguém vai avaliar a gramática)*

- **O que me surpreendeu.** ...
- **O que eu testaria a seguir.** ...

---
# Bloco 3 · Memória

### 🎯 Uma ideia
**O modelo não lembra de nada. Tudo o que parece memória é a lista de `messages` que você mandou de novo.**

Essa ideia contradiz a experiência de qualquer pessoa que já usou um chat. Você pergunta algo, ele responde, você pergunta "e por quê?" e ele sabe do que você está falando. Parece memória. Não é. A cada chamada, o servidor recebe a conversa **inteira**, do começo, e o modelo lê tudo de novo antes de gerar o próximo token. Quem guarda a conversa é o cliente, não o modelo. Isso tem três consequências que este bloco mostra na prática.

1. Uma chamada isolada não sabe nada sobre a anterior.
2. Uma conversa longa custa mais a cada turno, porque o prompt cresce.
3. O modelo só "lembra" o que cabe na janela de contexto que você viu no `/api/show`.

### 🔮 Preveja
O primeiro turno manda a mensagem de rádio. O segundo pergunta *"So how many laps do I have to hold on?"* **sem** mandar o primeiro. O que o modelo vai responder? Escolha uma aposta.

- (a) Vai responder sobre o pneu, porque acabou de ver a mensagem.
- (b) Vai dizer que não sabe do que se trata.
- (c) Vai inventar um contexto e responder com confiança.

### ▶️ Rode

In [ ]:
# Duas chamadas independentes. Nenhuma sabe da outra.
turn1 = client.chat.completions.create(
    model=SMALL, temperature=0,
    messages=[{"role": "user", "content": f"Reply as a race engineer, in one sentence: {RADIO}"}],
).choices[0].message.content
print("Turno 1 →", turn1.strip(), "\n")

turn2 = client.chat.completions.create(
    model=SMALL, temperature=0,
    messages=[{"role": "user", "content": "So how many laps do I have to hold on?"}],   # sem histórico
).choices[0].message.content
print("Turno 2 →", turn2.strip())

### 🔍 O que você deve ver

No turno 2 o modelo está respondendo no vazio. Dependendo do sorteio, ele pergunta do que você está falando (aposta b) ou, mais frequentemente com um modelo pequeno, **inventa** um contexto e responde com segurança sobre voltas que nunca foram mencionadas (aposta c). As duas reações são coerentes com a ideia do bloco. O modelo não está mentindo nem esquecendo. Ele simplesmente não recebeu o turno 1, e completar texto plausível é o único trabalho que ele sabe fazer.

Se você apostou em (a), ótimo. Errar essa previsão é a melhor forma de nunca mais esquecer que a memória está no cliente.

### 🧩 Preencha a lacuna

Complete `say()` para sustentar uma conversa de três turnos em que o modelo faz o papel de engenheiro de pista. A lista `history` precisa crescer a cada turno, e o *system prompt* define o personagem. O terceiro turno é a prova. Ele só pode ser respondido corretamente se o primeiro turno estiver na lista.

Aproveite e imprima `resp.usage.prompt_tokens` a cada turno. Esse número é o custo da memória.

✋ **Sua escolha.** Mantenha o `SYSTEM` sugerido ou escreva o seu (um engenheiro sarcástico, um que só fala em números, um que responde em português). O personagem muda o tom, mas a prova do turno 3 tem que continuar passando.

🧪 **Como saber se deu certo.** A resposta ao turno 3 precisa mencionar o pneu traseiro esquerdo (*rear left*). E `prompt_tokens` precisa **crescer** a cada turno, porque cada chamada reenvia tudo o que veio antes. Se em alguma execução um turno reportar um número menor que o anterior, não é a sua função. O servidor reaproveita em cache o começo do prompt e a contagem pode refletir isso. Rode de novo e olhe a tendência.

In [ ]:
SYSTEM = "You are a Formula 1 race engineer. Be brief, calm and concrete."
history = [{"role": "system", "content": SYSTEM}]

def say(text: str) -> str:
    # ---- SEU CÓDIGO AQUI ----
    # 1. acrescente o turno do usuário ao history
    # 2. chame o modelo com o history INTEIRO (temperature=0 ajuda a comparar execuções)
    # 3. acrescente o turno do assistente ao history
    # 4. imprima resp.usage.prompt_tokens e devolva o texto do assistente
    ...

print(say(RADIO), "\n")
print(say("So how many laps do I have to hold on?"), "\n")
print(say("Which tyre did I say was gone?"))   # esta linha é a prova

<details>
<summary><b>🔑 Solução de referência</b></summary>

```python
SYSTEM = "You are a Formula 1 race engineer. Be brief, calm and concrete."
history = [{"role": "system", "content": SYSTEM}]

def say(text: str) -> str:
    history.append({"role": "user", "content": text})
    resp = client.chat.completions.create(model=SMALL, messages=history, temperature=0)
    answer = resp.choices[0].message.content.strip()
    history.append({"role": "assistant", "content": answer})
    print(f"   [contexto enviado: {resp.usage.prompt_tokens} tokens]")
    return answer

print(say(RADIO), "\n")
print(say("So how many laps do I have to hold on?"), "\n")
print(say("Which tyre did I say was gone?"))   # esta linha é a prova
```

Repare no que a função faz e no que não faz. Ela **não** chama nenhuma API de "sessão" nem guarda um identificador de conversa no servidor. A conversa é uma lista Python. Se você reiniciar o kernel, a memória some, porque ela nunca esteve no modelo.

</details>

### 🔒 Fechamento
**Contexto é algo que você paga em toda chamada. Troque a `base_url` e esse mesmo laço fala com um modelo na nuvem. É por isso que a aula usa o SDK da OpenAI.**

### 🧭 Por que o experimento é assim
Este é o bloco em que a **continuidade** da Gestalt trabalha mais. A mensagem de rádio deixa de ser um ponto isolado e vira o primeiro elo de uma linha, e a prova do turno 3 só existe porque a linha foi mantida. Do lado da didática, repare no esqueleto de `say()`. Aqui você recebeu quatro passos numerados, como no Bloco 0. No Bloco 5 vai receber dois marcadores. No Bloco 7, uma linha. Os andaimes estão sendo retirados de propósito, um bloco por vez, para que a autonomia cresça sem que você caia.

### 🐇 Toca do coelho
Faça `history` crescer até estourar. Num laço, mande 40 mensagens curtas (`say(f"Lap {i}: still holding position.")`) e observe `prompt_tokens`. Em algum momento ele para de crescer ou a resposta do turno 3 deixa de funcionar. Você encontrou a janela de contexto (`num_ctx`, 4096 tokens por padrão nas versões recentes do Ollama, ajustável pela Porta 1 em `options`). O que o servidor faz com o começo da conversa quando ela não cabe mais? Ele corta, silenciosamente. Em produção, decidir **o que** cortar, ou o que buscar para colocar de volta, é o assunto da aula 08.

In [ ]:
# 🐇 Espaço livre para a toca do coelho. Nada aqui é obrigatório.

### 📓 Diário de bordo · Bloco 3

*(clique duas vezes nesta célula e escreva duas linhas; ninguém vai avaliar a gramática)*

- **O que me surpreendeu.** ...
- **O que eu testaria a seguir.** ...

---
# Bloco 4 · Modo JSON

> Se você está começando a segunda sessão de estudo, bem-vindo de volta. O Colab apagou a máquina virtual da sessão anterior, então rode de novo as três células de preparação (instalar, subir o servidor, baixar os modelos), as duas de configuração (instalação dos pacotes e constantes) e a primeira célula do Bloco 1 (que cria `client`). O resto deste bloco não depende de nada mais.

### 🎯 Uma ideia
**Software não quer um parágrafo. `response_format={"type": "json_object"}` obriga o modelo a emitir JSON que dá para fazer *parse*.**

Até aqui, o modelo respondeu com texto para uma pessoa ler. Mas o objetivo desta aula, e da disciplina, é colocar um modelo **dentro de um sistema**. Um sistema não lê parágrafos. Ele precisa de um campo `urgency` para decidir se toca um alarme e de um campo `topic` para rotear a mensagem. O primeiro passo é fazer o modelo falar a língua do software, JSON.

O modo JSON funciona no nível do decodificador. A cada token, o servidor consulta uma gramática do JSON e só permite os tokens que mantêm a saída válida. O texto que sai sempre fecha as chaves e as aspas. Mas há um detalhe, e a lacuna deste bloco existe para você descobri-lo.

### 🔮 Preveja
1. Quais chaves o modelo vai escolher para descrever a mensagem? Escreva três palpites.
2. Se você rodar duas vezes com temperatura zero, as chaves serão as mesmas? E se trocar para o modelo grande?

### ▶️ Rode

In [ ]:
SYSTEM_JSON = ("You are an API that answers ONLY in JSON. Given a radio message from a racing driver, "
               "describe what is happening and what the driver needs.")

resp = client.chat.completions.create(
    model=SMALL, temperature=0,
    messages=[{"role": "system", "content": SYSTEM_JSON},
              {"role": "user", "content": f"Radio message (answer in JSON): {RADIO}"}],
    response_format={"type": "json_object"},
)
raw = resp.choices[0].message.content
print("texto cru →", raw, "\n")

try:
    data = json.loads(raw)                                   # str -> dict
    print("dict Python →", data)
    print("chaves escolhidas pelo modelo →", list(data.keys()))
except json.JSONDecodeError as e:
    print("Não é JSON válido:", e)                           # raro em modo JSON, mas possível se a resposta for truncada

### 🔍 O que você deve ver

Um objeto JSON válido, com chaves que **o modelo inventou**. Algo como `situation`, `issue`, `request`, `driver_needs`, `tyre`. Rode de novo com `BIG` e as chaves mudam. Rode com temperatura 1 e mudam de novo. O `json.loads` funciona todas as vezes, e é exatamente isso que o modo JSON promete. Ele promete a **forma** (JSON válido). Não promete o **conteúdo** (quais chaves, quais tipos, quais valores).

Para o software que vai consumir essa resposta, chaves que mudam a cada chamada são tão inúteis quanto um parágrafo. Esse problema é o assunto do Bloco 5.

### 🧩 Preencha a lacuna

Copie a chamada acima e apague a palavra **JSON** dos dois prompts (o de sistema e o do usuário), mantendo o `response_format`. Coloque `max_tokens=200` como cinto de segurança e imprima `repr(raw)` em vez de `raw`, para enxergar espaços e quebras de linha. Rode três vezes. O que acontece? Depois coloque a palavra de volta e confirme que voltou ao normal.

Há uma regra escondida aqui. Encontre-a e escreva-a numa linha, no diário de bordo.

🧪 **Como saber se deu certo.** Não existe um resultado "certo" nesta lacuna, existe um fenômeno para observar. Os comportamentos comuns são uma resposta cheia de espaços e quebras de linha (por isso o `repr`), uma resposta que só tem `{}` ou um objeto quase vazio, uma resposta que demora muito mais que o normal (por isso o `max_tokens`), ou um JSON normal, se o modelo pequeno estiver num dia bom. Você encontrou a regra se conseguir explicar **por que** a gramática sozinha não basta.

In [ ]:
# ---- SEU CÓDIGO AQUI ----
# a mesma chamada, sem a palavra JSON nos prompts, com max_tokens=200 e print(repr(raw))

<details>
<summary><b>🔑 Solução de referência</b></summary>

```python
SYSTEM_NO_JSON = ("You are an API. Given a radio message from a racing driver, "
                  "describe what is happening and what the driver needs.")

for i in range(3):
    resp = client.chat.completions.create(
        model=SMALL, temperature=0.7, max_tokens=200,
        messages=[{"role": "system", "content": SYSTEM_NO_JSON},
                  {"role": "user", "content": f"Radio message: {RADIO}"}],
        response_format={"type": "json_object"},
    )
    print(f"[{i+1}]", repr(resp.choices[0].message.content))
```

**A regra.** A gramática do modo JSON só pode escolher entre os tokens que o modelo *já queria* emitir. Se o modelo, sem ninguém pedir JSON, quer começar a resposta com "The driver...", a gramática proíbe o `T`, proíbe o `h`, e o que sobra de permitido são chaves, aspas e espaços em branco. Um modelo que não foi instruído a produzir JSON frequentemente "escolhe" o espaço em branco, token após token, até o limite. A documentação do Ollama diz isso em uma frase, *it's important to instruct the model to use JSON in the prompt, otherwise the model may generate large amounts of whitespace*. A regra, portanto, é que **o prompt pede e a gramática garante, e um não substitui o outro**.

</details>

### 🔒 Fechamento
**Modo JSON garante a *forma* da saída, não o *conteúdo*. As chaves continuam sendo um palpite do modelo.**

### 🧭 Por que o experimento é assim
A lacuna deste bloco não pede código novo. Pede que você **quebre** uma coisa que funcionava e observe. É o princípio do **fechamento** virado do avesso. Em vez de completar uma lacuna, você a abre, e a mente precisa de uma regra para fechá-la de novo. Uma regra que você descobre observando um comportamento estranho é lembrada por muito mais tempo do que a mesma regra lida numa documentação, e é por isso que a documentação só aparece na solução, depois da observação.

### 🐇 Toca do coelho
Pela Porta 1, o modo JSON é `"format": "json"` no corpo do `POST /api/chat`. Faça a mesma chamada pelas duas portas e compare o texto cru. Depois, uma pergunta para quem gosta de gramáticas. O modo JSON aceita `{"a": 1}` e aceita `[1, 2, 3]`. Os dois são JSON. Qual dos dois o modelo escolhe quando o prompt não diz nada? O que isso revela sobre o que há nos dados de treinamento?

In [ ]:
# 🐇 Espaço livre para a toca do coelho. Nada aqui é obrigatório.

### 📓 Diário de bordo · Bloco 4

*(clique duas vezes nesta célula e escreva duas linhas; ninguém vai avaliar a gramática)*

- **O que me surpreendeu.** ...
- **O que eu testaria a seguir.** ...

---
# Bloco 5 · Contrato

### 🎯 Uma ideia
**Um modelo Pydantic é um contrato. O LLM assina ou o código recusa a resposta.**

No Bloco 4, o modelo escolhia as chaves. Agora você escolhe. A ferramenta é o Pydantic, a biblioteca que define, em uma classe Python, exatamente quais campos existem, de que tipo são e quais valores aceitam. Essa classe faz dois trabalhos ao mesmo tempo.

1. **Gera um schema** (`model_json_schema()`), um JSON que descreve o contrato e que você coloca no prompt para o modelo ler.
2. **Valida a resposta** (`model_validate_json()`), lendo o JSON que voltou e levantando `ValidationError` se algo não bate.

O segundo trabalho é o que muda o jogo. Uma resposta inválida deixa de ser um texto estranho no meio do sistema e vira uma **exceção**, com o nome do campo, o valor recebido e o motivo. Exceção você captura, registra, tenta de novo. Parágrafo, não.

### 🔮 Preveja
O contrato abaixo tem cinco campos. Na lacuna, você vai rodar o modelo pequeno 10 vezes com temperatura 0,8 e contar quantas respostas quebram o contrato.

1. Quantas das 10 vão falhar? Escreva um número.
2. Qual campo vai falhar mais? `urgency` fora de 1 a 5, `sentiment` com um valor fora da lista, `topic` com uma grafia diferente, ou um campo que simplesmente não veio?

### ▶️ Rode

In [ ]:
class RadioMessage(BaseModel):
    driver_request: str = Field(description="What the driver is asking for, in one sentence")
    topic: str                                            # vira um Literal na lacuna abaixo
    urgency: int = Field(ge=1, le=5, description="1 is calm, 5 is emergency")
    sentiment: Literal["calm", "frustrated", "angry"]
    requires_action: bool

SCHEMA = RadioMessage.model_json_schema()
print(json.dumps(SCHEMA, indent=2))

In [ ]:
SYSTEM_CONTRACT = ("You are an API that answers ONLY in JSON matching this schema exactly. "
                   "Do not add keys, do not omit keys.\n\n" + json.dumps(SCHEMA))

def extract_with_prompt(text: str, model: str = SMALL, temperature: float = 0.0) -> RadioMessage:
    """Bloco 5: o schema vai no PROMPT, modo JSON ligado, validação com Pydantic na volta."""
    resp = client.chat.completions.create(
        model=model, temperature=temperature,
        messages=[{"role": "system", "content": SYSTEM_CONTRACT},
                  {"role": "user", "content": f"Radio message: {text}"}],
        response_format={"type": "json_object"},
    )
    return RadioMessage.model_validate_json(resp.choices[0].message.content)

try:
    msg = extract_with_prompt(RADIO)
    print("✅ contrato assinado")
    print(msg.model_dump_json(indent=2))
except ValidationError as e:
    print(f"❌ contrato quebrado ({e.error_count()} erro(s))")
    for err in e.errors():
        print(f"   campo {err['loc']} → {err['msg']} | valor recebido: {err.get('input')!r}")

### 🔍 O que você deve ver

Primeiro, o schema. Leia-o como se fosse a primeira vez. `urgency` ganhou `minimum` e `maximum`. `sentiment` virou um `enum` com três valores. `description` virou texto que o modelo lê. Tudo o que você escreveu na classe está ali, traduzido para uma linguagem que o modelo entende. Escrever boas `description` é *prompt engineering* disfarçado de tipagem.

Depois, uma de duas coisas. Um ✅ com o objeto validado (o caso comum com temperatura zero), ou um ❌ com a lista de erros. Se aparecer o ❌, não corrija nada ainda. **Leia o erro.** Ele diz o campo (`loc`), o problema (`msg`) e o que o modelo mandou (`input`). Essa tríade é a diferença entre "o modelo alucinou" e "o modelo mandou `sentiment='urgent'`, que não está na lista".

### 🧩 Preencha a lacuna

Duas mudanças e uma medição.

1. Redefina `RadioMessage` trocando `topic: str` por `topic: Literal["tyres", "engine", "traffic", "strategy", "weather"]`. Recalcule `SCHEMA` e `SYSTEM_CONTRACT` (eles capturaram o schema antigo).
2. Rode `extract_with_prompt(RADIO, temperature=0.8)` dez vezes num laço, dentro de `try/except ValidationError`. Conte as falhas e guarde a **primeira** exceção inteira para imprimir ao final, exatamente como veio.

Por que temperatura 0,8 e não 0? Porque com temperatura zero as dez respostas seriam idênticas e a contagem só poderia dar 0 ou 10. A variação é o que você quer medir.

✋ **Sua escolha.** Acrescente um sexto campo ao contrato, que faça sentido para um pit wall (`tyre_position: Optional[Literal["front left", "front right", "rear left", "rear right"]]`, `laps_remaining_estimate: Optional[int]`, o que quiser). Campos opcionais precisam de um valor padrão (`= None`).

🧪 **Como saber se deu certo.** Com o modelo pequeno, espere entre 0 e 5 falhas em 10. Zero falhas é possível e não é erro. A falha mais comum é um valor fora do `Literal` (um `sentiment` como `"worried"`, um `topic` como `"tires"` com grafia americana). Repare que `urgency` vindo como texto `"4"` **não** falha, porque o Pydantic converte `"4"` em `4` no modo padrão (*lax*). Se você quiser que falhe, `ConfigDict(strict=True)` resolve, e essa é uma decisão de contrato, não do modelo.

In [ ]:
class RadioMessage(BaseModel):
    driver_request: str = Field(description="What the driver is asking for, in one sentence")
    # ---- SEU CÓDIGO AQUI (1) ----  topic vira Literal com cinco valores
    topic: str
    urgency: int = Field(ge=1, le=5, description="1 is calm, 5 is emergency")
    sentiment: Literal["calm", "frustrated", "angry"]
    requires_action: bool

SCHEMA = RadioMessage.model_json_schema()
SYSTEM_CONTRACT = ("You are an API that answers ONLY in JSON matching this schema exactly. "
                   "Do not add keys, do not omit keys.\n\n" + json.dumps(SCHEMA))

failures, first_error = 0, None
for i in range(10):
    # ---- SEU CÓDIGO AQUI (2) ----  try / except ValidationError, conte e guarde a primeira
    ...

print(f"{failures}/10 respostas quebraram o contrato")
if first_error:
    print("\nprimeira falha, na íntegra:\n", first_error)

<details>
<summary><b>🔑 Solução de referência</b></summary>

```python
class RadioMessage(BaseModel):
    driver_request: str = Field(description="What the driver is asking for, in one sentence")
    topic: Literal["tyres", "engine", "traffic", "strategy", "weather"]
    urgency: int = Field(ge=1, le=5, description="1 is calm, 5 is emergency")
    sentiment: Literal["calm", "frustrated", "angry"]
    requires_action: bool

SCHEMA = RadioMessage.model_json_schema()
SYSTEM_CONTRACT = ("You are an API that answers ONLY in JSON matching this schema exactly. "
                   "Do not add keys, do not omit keys.\n\n" + json.dumps(SCHEMA))

failures, first_error = 0, None
for i in range(10):
    try:
        extract_with_prompt(RADIO, temperature=0.8)
    except ValidationError as e:
        failures += 1
        first_error = first_error or e

print(f"{failures}/10 respostas quebraram o contrato")
if first_error:
    print("\nprimeira falha, na íntegra:\n", first_error)
```

Note que `extract_with_prompt` não precisou mudar. Ela lê `RadioMessage` e `SYSTEM_CONTRACT` do escopo global no momento da chamada, então redefinir os dois foi suficiente. Se você preferir código sem esse tipo de dependência escondida, passe o contrato como argumento da função. É uma escolha de engenharia legítima e vale uma linha no diário.

</details>

### 🔒 Fechamento
**Validação transforma alucinação em exceção. Exceção você captura. Parágrafo, não.**

### 🧭 Por que o experimento é assim
Compare o esqueleto desta lacuna com o do Bloco 3. Lá, quatro passos numerados. Aqui, dois marcadores e a instrução em prosa. Os **andaimes** foram encurtados de propósito, e a ✋ escolha (um sexto campo) é a primeira em que você altera o contrato do sistema, não só um parâmetro. Do lado da Gestalt, o contrato é **Prägnanz** em estado puro. Uma classe de cinco linhas é a descrição mais simples e completa do que o software precisa, e é lida de uma vez, como uma forma, e não campo por campo.

### 🐇 Toca do coelho
**O experimento em português.** Troque `RADIO` por `RADIO_PT` (definida na configuração) e repita as 10 rodadas. Contam-se as falhas e comparam-se os `driver_request` gerados. Duas coisas costumam aparecer. O modelo pequeno erra mais o contrato em português, e às vezes responde `driver_request` em inglês mesmo lendo português, porque a instrução do sistema está em inglês. Volte ao relatório da aula 06 e confira quantos tokens a mesma mensagem custa em cada língua. Custo e qualidade caminham juntos aqui, e nos dois casos o português paga mais.

In [ ]:
# 🐇 Espaço livre para a toca do coelho. Nada aqui é obrigatório.

### 📓 Diário de bordo · Bloco 5

*(clique duas vezes nesta célula e escreva duas linhas; ninguém vai avaliar a gramática)*

- **O que me surpreendeu.** ...
- **O que eu testaria a seguir.** ...

---
# Bloco 6 · Garantia

### 🎯 Uma ideia
***Structured outputs* restringem a própria decodificação. O modelo não consegue emitir um token que quebre o schema, e a validação para de falhar.**

O Bloco 5 colocou o schema no prompt, e o prompt é um **pedido**. O modelo lê, entende na maior parte das vezes, e de vez em quando escreve `"worried"` onde só cabia `"calm"`, `"frustrated"` ou `"angry"`. Neste bloco o schema vai para outro lugar, o **decodificador**. O servidor transforma o schema numa gramática (a mesma técnica do modo JSON, só que muito mais específica) e, a cada token, só deixa passar o que mantém a saída dentro do schema. Depois de `"sentiment": "`, os únicos tokens permitidos são os que começam `calm`, `frustrated` ou `angry`. O modelo não tem como errar, porque a opção errada nunca é oferecida.

Isso não é mágica e tem um custo, que a lacuna vai medir. Também não resolve tudo. Um valor **válido** ainda pode ser **errado** (um `urgency` de 2 para um pneu destruído é válido para a gramática). Por isso a validação com Pydantic continua no código, agora como cinto e suspensório.

### 🔮 Preveja
1. Com *structured outputs*, quantas de 20 respostas vão quebrar o contrato?
2. A latência média por chamada vai ser menor, igual ou maior do que a do Bloco 5? Por quê?

### ▶️ Rode

In [ ]:
def extract_with_schema(text: str, model: str = SMALL, temperature: float = 0.0) -> RadioMessage:
    """Bloco 6: o schema vai para o DECODIFICADOR (structured outputs). O prompt fica curto."""
    resp = client.chat.completions.create(
        model=model, temperature=temperature,
        messages=[{"role": "system", "content": "You are an API that extracts information from F1 radio messages."},
                  {"role": "user", "content": f"Radio message: {text}"}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": "radio", "schema": RadioMessage.model_json_schema()}},
    )
    return RadioMessage.model_validate_json(resp.choices[0].message.content)

try:
    msg = extract_with_schema(RADIO)
    print(msg.model_dump_json(indent=2))
except ValidationError as e:           # cinto e suspensório: a gramática garante a forma, o código ainda confere
    print("O contrato falhou mesmo com structured outputs. Leia o erro com calma, a causa quase sempre é uma resposta truncada.")
    print(e)

# Plano B (Porta 1): a mesma garantia pela API nativa. O schema vai no campo "format".
# payload = {"model": SMALL, "stream": False, "format": RadioMessage.model_json_schema(),
#            "messages": [{"role": "user", "content": f"Radio message: {RADIO}"}]}
# print(requests.post(f"{OLLAMA_URL}/api/chat", json=payload).json()["message"]["content"])

### 🔍 O que você deve ver

Um objeto com **exatamente** as chaves do contrato, na ordem do contrato, com `topic` e `sentiment` dentro das listas permitidas. Repare que o *system prompt* encolheu para uma frase. O schema não precisa mais ser explicado ao modelo, porque ele não é mais uma instrução, é uma restrição.

Se `RadioMessage` ainda tiver `topic: str` (porque você pulou a lacuna do Bloco 5), a célula funciona do mesmo jeito, só que `topic` fica livre. Vale a pena voltar e fazer a troca. A comparação da lacuna abaixo fica muito mais interessante com o `Literal`.

### 🧩 Preencha a lacuna

Rode as duas abordagens 20 vezes cada, sobre `RADIO`, no modelo `SMALL`, com `temperature=0.8` (pelo mesmo motivo do Bloco 5). Para cada uma, conte as falhas de validação e a latência média. Monte um DataFrame com uma linha por abordagem. Depois responda numa linha, no diário, **por que o Bloco 6 não é de graça**.

| abordagem | falhas / 20 | latência média (s) |
|---|---|---|
| prompt + validação (Bloco 5) | | |
| structured outputs (Bloco 6) | | |

✋ **Sua escolha.** Meça também no modelo `BIG` (com GPU são segundos, em CPU são alguns minutos) e acrescente duas linhas à tabela. A pergunta que vale ouro é se a garantia importa *menos* num modelo maior.

🧪 **Como saber se deu certo.** A linha do Bloco 6 deve ter **zero** falhas. Se aparecer alguma, olhe o erro. Ou a resposta foi truncada pelo limite de tokens no meio de um texto longo, ou o contrato tem uma regra que a gramática não consegue expressar (uma instrução escrita em `description`, uma validação que envolve dois campos). Descobrir qual das duas é, por si só, um resultado. A latência do Bloco 6 costuma ser parecida ou um pouco maior, e a **primeira** chamada com um schema novo é visivelmente mais lenta, porque o servidor compila a gramática e guarda em cache.

In [ ]:
def benchmark(fn, n: int = 20, **kwargs) -> dict:
    """Roda fn(RADIO, **kwargs) n vezes e devolve falhas de validação e latência média."""
    failures, elapsed = 0, []
    for _ in range(n):
        t0 = time.time()
        # ---- SEU CÓDIGO AQUI ----
        ...
        elapsed.append(time.time() - t0)
    return {"falhas": f"{failures}/{n}", "latência média (s)": sum(elapsed) / len(elapsed)}

results = pd.DataFrame({
    "prompt + validação (Bloco 5)": benchmark(extract_with_prompt, temperature=0.8),
    "structured outputs (Bloco 6)": benchmark(extract_with_schema, temperature=0.8),
}).T
results.round(2)

<details>
<summary><b>🔑 Solução de referência</b></summary>

```python
def benchmark(fn, n: int = 20, **kwargs) -> dict:
    failures, elapsed = 0, []
    for _ in range(n):
        t0 = time.time()
        try:
            fn(RADIO, **kwargs)
        except ValidationError:
            failures += 1
        elapsed.append(time.time() - t0)
    return {"falhas": f"{failures}/{n}", "latência média (s)": sum(elapsed) / len(elapsed)}

results = pd.DataFrame({
    "prompt + validação (Bloco 5)": benchmark(extract_with_prompt, temperature=0.8),
    "structured outputs (Bloco 6)": benchmark(extract_with_schema, temperature=0.8),
}).T
results.round(2)
```

**Por que não é de graça.** Três motivos, em ordem de importância. Primeiro, a gramática precisa ser compilada e consultada a cada token, o que custa tempo, pouco em schemas simples e muito em schemas grandes ou recursivos. Segundo, a garantia é sobre a forma, e uma forma garantida convida a **confiar demais** no conteúdo. Um `urgency: 2` para um pneu destruído passa na gramática e passa no Pydantic. Só um teste com dados rotulados pega esse erro, e é isso que o Bloco 7 e o exercício da semana fazem. Terceiro, forçar o modelo a escolher entre cinco tópicos quando a mensagem não pertence a nenhum deles produz uma resposta confiante e errada. Um contrato bem desenhado prevê a saída `"other"`.

</details>

### 🔒 Fechamento
**Schema no prompt é um pedido. Schema no decodificador é uma garantia. A velha afirmação de que isso só funciona na nuvem já não é verdade.**

### 🧭 Por que o experimento é assim
Este bloco é construído sobre **contraste**. A mesma mensagem, o mesmo modelo, a mesma temperatura, o mesmo número de rodadas, e duas abordagens lado a lado numa tabela de duas linhas. Casos contrastantes (Schwartz e Bransford chamaram isso de *a time for telling*) preparam a pessoa para entender a explicação que vem depois. Você mede primeiro e lê o "por que não é de graça" depois, e a explicação encaixa porque a tabela já abriu o espaço para ela. Na Gestalt, é **similaridade** de novo. Duas linhas com as mesmas colunas são percebidas como um par, e a diferença salta.

### 🐇 Toca do coelho
Duas ideias.

1. **O classificador mais barato do mundo.** Um schema com um único campo `enum` obriga o modelo a responder com **um** token, praticamente sem custo de geração. Monte `{"type": "object", "properties": {"topic": {"enum": [...]}}, "required": ["topic"]}` e meça a latência. Compare com o Bloco 5. Isso é o que a aula 06 chamou de "ler a distribuição" (Q4 do relatório), agora pela API.
2. **De onde vem a gramática.** O Ollama usa o motor do `llama.cpp`, que converte JSON Schema em uma gramática GBNF. Procure `json_schema_to_grammar` no repositório do `llama.cpp` e leia a gramática gerada para o seu `RadioMessage`. Ela cabe numa tela, e é tudo o que separa um pedido de uma garantia.

In [ ]:
# 🐇 Espaço livre para a toca do coelho. Nada aqui é obrigatório.

### 📓 Diário de bordo · Bloco 6

*(clique duas vezes nesta célula e escreva duas linhas; ninguém vai avaliar a gramática)*

- **O que me surpreendeu.** ...
- **O que eu testaria a seguir.** ...

---
# Bloco 7 · Escala

### 🎯 Uma ideia
**Uma mensagem é um exemplo. Trinta mensagens são um dataset, e um dataset ganha um gráfico (aula 04).**

Tudo o que você fez até aqui foi com uma única mensagem. Isso foi de propósito, para que cada ideia coubesse na cabeça de uma vez. Mas um sistema de verdade não vê uma mensagem, vê um fluxo. Neste bloco a função `extract_with_schema` do Bloco 6 roda sobre um arquivo com 30 mensagens de rádio de uma corrida inteira, dois carros (27 e 88), 57 voltas, e o resultado vira um DataFrame. A partir daí, as ferramentas são as da aula 04. `head()`, `groupby`, um gráfico com uma pergunta clara.

O arquivo traz, além de `lap`, `driver_id` e `text`, três colunas de **referência** (`topic_ref`, `urgency_ref`, `sentiment_ref`), rotuladas por uma pessoa. Elas são a resposta que o modelo não vê e que você vai usar para medir quanto ele acerta. É o mesmo papel dos 30 exemplos rotulados do relatório da aula 06.

### 🔮 Preveja
1. Com os tokens por segundo que você mediu no Bloco 2, quanto tempo as 30 mensagens vão levar no modelo pequeno? Faça a conta.
2. Em quantas das 30 o `topic` do modelo vai discordar do rótulo humano?
3. Qual tipo de mensagem o modelo de 0,5B vai errar mais? Uma irônica, uma com dois assuntos, uma que exige saber o que é *blue flag*, ou uma curta demais?

### ▶️ Rode

In [ ]:
# O dataset. Se o arquivo não estiver ao lado do notebook, baixa do repositório da disciplina.
CSV = "radio_messages.csv"
if not os.path.exists(CSV):
    url = "https://raw.githubusercontent.com/ivanovitchm/aiengineering/main/lessons/week05/radio_messages.csv"
    open(CSV, "wb").write(requests.get(url).content)

radio = pd.read_csv(CSV)
print(f"{len(radio)} mensagens · colunas {list(radio.columns)}")
radio.head(6)

In [ ]:
def classify(text: str, model: str = SMALL) -> Optional[dict]:
    """Bloco 6 aplicado a uma mensagem. Devolve um dict ou None se o contrato falhar."""
    try:
        return extract_with_schema(text, model=model).model_dump()
    except ValidationError:
        return None

EMPTY = {field: None for field in RadioMessage.model_fields}   # garante as colunas mesmo quando o contrato falha

t0 = time.time()
rows = []
for r in radio.itertuples(index=False):
    out = classify(r.text)
    rows.append({"lap": r.lap, "driver_id": r.driver_id, "text": r.text,
                 **(out or EMPTY), "ok": out is not None})

df = pd.DataFrame(rows)
df[["topic_ref", "urgency_ref", "sentiment_ref"]] = radio[["topic_ref", "urgency_ref", "sentiment_ref"]]
print(f"{len(df)} mensagens em {time.time() - t0:.0f} s · contratos válidos {df.ok.sum()}/{len(df)}")
df.head()

### 🔍 O que você deve ver

Um DataFrame com uma linha por mensagem, as colunas do contrato preenchidas pelo modelo e as três colunas de referência ao lado. Procure a volta 41 do carro 27. É a mensagem que abriu o notebook, agora sentada ao lado das outras 29, como prometido na primeira célula.

Compare o tempo total com a sua conta do 🔮. Se demorou mais do que a conta, lembre que cada chamada paga o *prefill* do prompt e o *overhead* do servidor, e não só a geração. Esse é o número que importa para um sistema que precisa processar mensagens em tempo real.

### 🧩 Preencha a lacuna

Duas partes, e desta vez o esqueleto é só uma linha.

1. **O gráfico.** Urgência ao longo das voltas, separada por tópico, seguindo as regras da aula 04 (uma pergunta por gráfico, rótulos diretos, sem *chart junk*, eixos com nome e unidade). Destaque a volta 41. A pergunta que o gráfico responde é "em que fase da corrida a urgência subiu, e por qual assunto?".
2. **O erro.** Liste as mensagens em que `topic` do modelo discorda de `topic_ref`. Escolha **uma** e explique, numa frase, *por que* um modelo de 0,5B erra exatamente aquela. A resposta não é "porque ele é pequeno". A resposta está no texto da mensagem.

✋ **Sua escolha.** Duas decisões são suas. A primeira é a forma do gráfico, uma linha por tópico num único painel ou um painel por tópico (*small multiples*), e o critério é qual das duas responde à pergunta sem rótulos se sobrepondo. A segunda é o que fazer depois, repetir a análise de erro para `urgency` (a discordância pode ser medida como distância, não só como acerto ou erro) ou para `sentiment`, ou rodar as 30 mensagens no modelo `BIG` e comparar os dois lado a lado. Qualquer uma dessas escolhas é um ensaio do exercício da semana.

🧪 **Como saber se deu certo.** O gráfico tem cinco linhas, cinco cores ou cinco painéis, um eixo y de 1 a 5, e dá para responder olhando "em que fase da corrida a urgência subiu, e por qual motivo?". A concordância de `topic` com o modelo pequeno costuma ficar entre 60% e 85%. Se der 100%, confira se você não está comparando `topic_ref` com ela mesma. Se der abaixo de 40%, confira se `RadioMessage` tem o `Literal` de cinco tópicos.

In [ ]:
# ---- SEU CÓDIGO AQUI ----
# Um painel só (uma linha por tópico) ou cinco painéis (um por tópico)? Decida antes de chamar plt.subplots.

<details>
<summary><b>🔑 Solução de referência</b></summary>

```python
# O estilo da aula 04: grades sutis, sem molduras, título à esquerda.
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.titlelocation": "left",
                     "axes.titleweight": "bold"})
C_MAIN, C_ACCENT = "#1f77b4", "#ff7f0e"      # um azul para os dados, um laranja para o destaque

ok = df[df.ok].sort_values("lap")
topics = ["tyres", "engine", "traffic", "strategy", "weather"]

# Um painel por tópico (small multiples): a mesma escala em todos, nenhum rótulo se sobrepõe.
fig, axes = plt.subplots(len(topics), 1, figsize=(9, 8), sharex=True)
for ax, topic in zip(axes, topics):
    g = ok[ok.topic == topic]
    ax.plot(g.lap, g.urgency, marker="o", color=C_MAIN, linewidth=1.2)
    ax.set_title(f"{topic}  ({len(g)} mensagens)", fontsize=10)
    ax.set_xlim(0, 58)
    ax.set_ylim(0.5, 5.5)
    ax.set_yticks([1, 3, 5])

# A mensagem que abriu o notebook, em destaque no painel do tópico que o modelo escolheu para ela.
row = ok[(ok.lap == 41) & (ok.driver_id == 27)]
if len(row):
    ax = axes[topics.index(row.topic.iloc[0])]
    ax.scatter(41, row.urgency.iloc[0], s=180, facecolors="none", edgecolors=C_ACCENT, linewidths=2, zorder=5)
    ax.annotate("a mensagem que abriu o notebook", (41, row.urgency.iloc[0]),
                xytext=(-160, -22), textcoords="offset points", fontsize=9, color=C_ACCENT,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.85),
                arrowprops=dict(arrowstyle="-", color=C_ACCENT, lw=0.8))

axes[-1].set_xlabel("volta")
fig.supylabel("urgência (1 = calmo, 5 = emergência)")
fig.suptitle(f"Em que fase da corrida a urgência subiu, e por qual assunto?  ({SMALL})",
             x=0.01, ha="left", fontweight="bold")
plt.tight_layout()
plt.show()

# Onde o modelo discordou da pessoa que rotulou
agree = (df.topic == df.topic_ref).mean()
print(f"concordância em topic: {agree:.0%}")
wrong = df.loc[df.topic != df.topic_ref, ["lap", "driver_id", "topic_ref", "topic", "text"]]
display(wrong)
print(pd.crosstab(df.topic_ref, df.topic.fillna("inválido")))
```

**Sobre o gráfico.** Uma linha por tópico num painel só é a escolha mais óbvia, e produz um emaranhado em que os rótulos das cinco linhas caem uns sobre os outros. Painéis pequenos com a mesma escala (*small multiples*, uma ideia de Tufte) resolvem isso sem legenda, e o olho compara as fases da corrida descendo pelas linhas. É a regra "uma pergunta por gráfico" da aula 04 aplicada cinco vezes.

**Sobre o erro.** As mensagens que um modelo pequeno mais erra são as que exigem algo além do texto. *"Blue flags, blue flags! He's not moving over"* é `traffic`, mas só para quem sabe que bandeira azul é a ordem para um retardatário dar passagem. *"Vibration from the rear... is it the tyre or the gearbox?"* menciona dois assuntos e o modelo precisa escolher o dominante. *"Great strategy, guys. Really. Brilliant."* é ironia, e ironia é um dos últimos fenômenos que modelos pequenos aprendem. Uma frase de explicação boa aponta o mecanismo, não o tamanho do modelo.

</details>

### 🔒 Fechamento
**Um modelo local, um schema e um laço já são um pipeline de dados. Tudo depois disso é engenharia.**

### 🧭 Por que o experimento é assim
Este é o fechamento do arco de **continuidade**. A mensagem que entrou num `curl` no Bloco 0 é agora uma linha entre trinta, e o gráfico que a contém foi feito com as regras que você aprendeu na aula 04 sobre dados de outro domínio. Quando uma ferramenta antiga (o DataFrame, o `plt.subplots`) reaparece sobre um material novo (texto classificado por um modelo), ela é reorganizada na memória num nível mais alto. É a aprendizagem significativa de Ausubel em ação, e é o motivo de a aula terminar com um gráfico e não com uma chamada de API. A lacuna com esqueleto de uma linha é o último degrau da retirada dos andaimes. Daqui em diante, é o exercício da semana.

### 🐇 Toca do coelho
Rode as 30 mensagens no modelo `BIG` e monte `pd.crosstab(df_small.topic, df_big.topic)`. Onde os dois modelos discordam entre si, quem tem razão, segundo `topic_ref`? Isso é um ensaio do exercício da semana, e é também a técnica mais barata que existe para encontrar exemplos difíceis num dataset. Onde dois modelos discordam, vale a pena uma pessoa olhar.

In [ ]:
# 🐇 Espaço livre para a toca do coelho. Nada aqui é obrigatório.

### 📓 Diário de bordo · Bloco 7

*(clique duas vezes nesta célula e escreva duas linhas; ninguém vai avaliar a gramática)*

- **O que me surpreendeu.** ...
- **O que eu testaria a seguir.** ...

---
# 📝 Auto-teste final (recuperação espaçada)

Responda **sem rolar o notebook** e, de preferência, um ou dois dias depois de terminar os blocos. Depois confira. Errar agora e corrigir é o que fixa o conteúdo. Cada pergunta corresponde a um 🔒 que você disse em voz alta.

1. Em uma frase, o que é o Ollama e o que ele não é?
2. Quais são as duas portas para falar com o servidor, e qual delas você usaria para medir tokens por segundo? Por quê?
3. `temperature=0` e `seed=42`. Qual dos dois controla diversidade e qual controla reprodutibilidade?
4. O que `eval_count` e `eval_duration` medem, e em que unidade vem a duração?
5. Por que `prompt_tokens` cresce a cada turno de uma conversa? Onde a conversa fica guardada?
6. O que o modo JSON garante e o que ele não garante? E qual é a regra sobre a palavra "JSON" no prompt?
7. Quais são os dois trabalhos de um modelo Pydantic no pipeline?
8. Qual é a diferença entre colocar o schema no prompt e colocar o schema no decodificador? Cite um custo do segundo.
9. Por que a validação com Pydantic continua no código mesmo com *structured outputs*?
10. Escreva de memória o laço que transforma um CSV de mensagens num DataFrame classificado (quatro linhas bastam).

<details>
<summary><b>Clique para conferir um gabarito resumido</b></summary>

1. Um servidor HTTP local que carrega modelos e responde em JSON. Não é uma biblioteca Python.
2. HTTP cru (`requests` em `/api/...`) e o SDK da OpenAI com `base_url` local. Para velocidade, a Porta 1, porque só a API nativa devolve `eval_count` e `eval_duration`.
3. Temperatura controla diversidade. `seed` controla reprodutibilidade. Com `seed` fixo, até temperatura 1 repete a resposta.
4. Tokens gerados e o tempo gasto para gerá-los, em nanossegundos. Tokens por segundo é `eval_count / (eval_duration / 1e9)`.
5. Porque cada chamada reenvia a conversa inteira. A conversa fica no cliente, numa lista de `messages`, nunca no modelo.
6. Garante JSON sintaticamente válido. Não garante as chaves, os tipos nem os valores. Sem a palavra JSON no prompt, o modelo pode gerar espaços em branco até o limite, porque a gramática só escolhe entre o que o modelo já queria emitir.
7. Gerar o schema (`model_json_schema`) e validar a resposta (`model_validate_json`), transformando resposta inválida em `ValidationError`.
8. No prompt é um pedido, e o modelo pode ignorar. No decodificador é uma restrição, e o token errado nunca é oferecido. Os custos são compilar e consultar a gramática a cada token, e a tentação de confiar num conteúdo que é válido mas errado.
9. Porque nem toda regra de um contrato cabe numa gramática (uma instrução em `description`, uma validação entre campos), porque uma resposta pode ser truncada pelo limite de tokens, e porque o código deve continuar correto se alguém trocar o servidor por um que não suporte *structured outputs*.
10. `radio = pd.read_csv(...)` · `rows = [classify(r.text) for r in radio.itertuples()]` · `df = pd.DataFrame(rows)` · `df["topic_ref"] = radio.topic_ref` (ou equivalente).

</details>

---
# 🏁 Exercício da semana · Do notebook ao pit wall

Na semana passada você escreveu um *model selection report* olhando modelos no Hugging Face. Nesta semana o relatório ganha números de um modelo que roda de verdade, sobre um dataset de 200 mensagens, e a pergunta é a mesma que uma equipe faria.

> **Qual modelo você colocaria no carro, e por quê?**

### Construir

1. Escreva um `Modelfile` com um `SYSTEM` que transforme o modelo base num engenheiro de pista e com pelo menos duas linhas `PARAMETER` (`temperature`, `num_ctx`, ...). Um Modelfile é a receita de um modelo (toca do coelho do Bloco 0). Ele "assa" o *system prompt* e os parâmetros dentro de um novo nome, de modo que nenhuma chamada precise reenviá-los. Comece por `!ollama show --modelfile qwen2.5:0.5b` para ver a receita do modelo base. No Colab, o arquivo é escrito por uma célula e o modelo é criado pela seguinte.

   ```
   %%writefile Modelfile
   FROM qwen2.5:0.5b
   SYSTEM """You are the race engineer of car 27. ... """
   PARAMETER temperature 0.2
   PARAMETER num_ctx 4096
   ```

   ```
   !ollama create pitwall-small -f Modelfile          # e um pitwall-big sobre o qwen2.5:3b
   !ollama run pitwall-small "Rear left is gone."       # um teste manual antes de qualquer laço
   ```

   O modelo criado vive na máquina virtual e some com a sessão. O que se guarda no repositório é o `Modelfile`, e o notebook deve recriar os dois `pitwall` toda vez que roda, logo depois da preparação.

2. Rode as 200 mensagens de `radio_messages_200.csv` (mesmas colunas do arquivo de 30, no repositório da disciplina) pelos dois modelos, usando a abordagem do Bloco 6. Guarde tudo em um DataFrame por modelo. Com a GPU T4 são poucos minutos. Em CPU, o modelo grande pode levar mais de vinte, então planeje a sessão. Um detalhe que custa uma tarde a quem não sabe. Se a chamada enviar uma mensagem com `role: system`, ela **substitui** o `SYSTEM` do Modelfile. Para que o seu `pitwall` use o personagem que você escreveu, chame-o sem mensagem de sistema.

### Medir (uma tabela, os dois modelos)

| métrica | como |
|---|---|
| Taxa de contratos válidos | Bloco 5 e 6. Mesmo com *structured outputs*, conte. |
| Tokens por segundo | Bloco 2. Mediana de pelo menos três medições, com o modelo já carregado. Diga se mediu em CPU ou em GPU T4. |
| Concordância com os rótulos humanos | `topic` (acerto ou erro) e `urgency` (distância média em relação a `urgency_ref`). |
| Concordância entre os dois modelos | `topic` e `urgency`. Onde eles discordam, quem está certo segundo a referência? |

### Relatório (uma página, PDF, em inglês)

Responda à pergunta. Use as quatro métricas. **Uma frase sem número não conta.** Inclua um gráfico feito com as regras da aula 04 (a matriz de confusão de `topic` de cada modelo é uma boa escolha). Termine com um parágrafo sobre o que a tabela **não** mede e que você precisaria medir antes de colocar o modelo num carro.

### Bônus · O engenheiro está respondendo às cegas

Repare que, em todos os blocos, o modelo responde sem saber a volta, a posição ou quantos carros faltam. A API Jolpica (`https://api.jolpi.ca/ergast/f1/`) é gratuita, não exige chave e conhece cada volta, posição e parada de cada corrida real. Coloque a volta atual e a posição dentro do prompt do `pitwall` e mostre **uma** troca em que esse contexto muda a resposta. Guarde a palavra *retrieval*. Ela volta na aula 08.

### Entrega

Notebook executado, `Modelfile` e o PDF, no repositório da disciplina, pasta `lesson07/`. Individual. Prazo, a próxima aula.

### Como o trabalho é avaliado

| Critério | Peso | O que se espera |
|---|---|---|
| Os dois `pitwall` existem e o laço roda sobre as 200 mensagens | 25% | Código reproduzível, sem células quebradas |
| A tabela de métricas está completa e o método de medição está explicado | 30% | Mediana e não uma medição isolada; concordância calculada a partir de `_ref` |
| A decisão é defendida com números | 30% | Cada afirmação aponta para uma linha da tabela ou para o gráfico |
| Limitações e o que faltou medir | 15% | Pelo menos duas limitações concretas do experimento |
| Bônus (retrieval) | +10% | Uma troca antes e depois do contexto, com a diferença comentada |

### 🧭 Por que o exercício é assim
Os critérios estão na mesa antes de você começar, e cada um deles foi ensaiado num bloco (o Modelfile está na toca do coelho do Bloco 0, as métricas são os Blocos 2, 5 e 6, a concordância é o Bloco 7). Você escolhe o `SYSTEM`, os parâmetros e o que dizer sobre as limitações. Autonomia, aqui, é ter todas as ferramentas e decidir como usá-las, não adivinhar o que o professor quer.

---
# 🗺️ Onde tudo se conecta

| Você viu | Em | Volta como |
|---|---|---|
| DataFrames e gráficos limpos | Aula 04 | Bloco 7 e a tabela do exercício |
| Tokens, amostragem, temperatura | Aula 06 | Blocos 1 e 2 |
| A contagem de parâmetros à mão | Aula 06 | Bytes por parâmetro, Bloco 0 |
| SmolLM2-360M e Qwen2.5-0.5B | Aula 06 | Os modelos que você baixou hoje |
| Um modelo como servidor HTTP | Hoje | Toda API que você vai chamar daqui em diante |
| A conversa que vive no cliente | Bloco 3 | Todo *chatbot* que você construir |
| Um contrato entre modelo e software | Blocos 5 e 6 | Toda integração de LLM que vá para produção |
| Contexto dentro do prompt | Bônus do exercício | Aula 08, *retrieval* |

